In [1]:
# Disable ipywidgets progress bars to avoid context issues
import os

from collections import defaultdict
import logging
import json
import argparse

from datasets import load_dataset, Dataset
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader as TorchDataLoader
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from transformers.trainer_utils import set_seed

from src.activation_capture import ActivationCapture, Hook
from src.utilities.device_utils import get_device, empty_cache
from src.models.opt.activation_capture_opt import ActivationCaptureOpt

from measure_gt_sparsity2 import ContextualSparsityAnalyzer,analyze_sparsity,C4Dataset

In [2]:
from argparse import Namespace
args = Namespace()
args.device = "cuda"
args.dataset_name="allenai/c4"
args.dataset_config="en"
args.seed=44
args.num_samples=1000
args.max_length=256
args.batch_size=1
args.num_workers=8


threshold_file="thresholds.json"
with open(threshold_file, mode="r", encoding="utf-8") as read_file:
    thresholds = json.load(read_file)

device = get_device(args.device)
set_seed(args.seed)

In [ ]:
sparsity,diffcounts = analyze_sparsity(args, "facebook/opt-2.7b", device, None)

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

INFO:measure_gt_sparsity2:Loading C4 dataset...


README.md: 0.00B [00:00, ?B/s]

In [ ]:
out

In [ ]:
models = [
    "meta-llama/Llama-3.2-1B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "google/gemma-3n-E2B",
    "Qwen/Qwen2-1.5B",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "facebook/opt-2.7b"
]
outs = defaultdict(dict)
diffs = defaultdict(dict)
for model in models:
    out,diff = analyze_sparsity(args, model, device, thresholds.get(model, None))
    outs[model] = out
    diffs[model] = diff


INFO:measure_gt_sparsity2:Loading model: meta-llama/Llama-3.2-1B-Instruct


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/shadeform/sparse_transformers/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may hav

In [ ]:
outs

In [ ]:
out # llama 3.2 1b

In [ ]:
from collections import Counter
class ContextualSparsityAnalyzer:
    """Analyzer for measuring contextual sparsity patterns in LLaMA models."""

    def __init__(self, model, tokenizer, device, thresholds=[]):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.thresholds = thresholds

        model.activation_capture = capture_model(model)
        model.activation_capture.register_hooks(hooks=[Hook.ACT])
        self.num_layers = len(self.model.activation_capture.get_layers())

        self.reset_buffers()

    def reset_buffers(self):
        self.mlp_sparsity = defaultdict(list)
        self.heavy_hitters = defaultdict(Counter)
        self.num_seqs = 0

    def process_batch(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        batch_size = input_ids.size(0)

        # Clear previous captures and GPU cache
        self.model.activation_capture.clear_captures()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()

        # Forward pass
        with torch.no_grad():
            _ = self.model(input_ids=input_ids, attention_mask=attention_mask)

        # Compute sparsity
        for layer_idx in range(self.num_layers):
            activations = self.model.activation_capture.mlp_activations[Hook.ACT][layer_idx]
            #threshold = self.thresholds[str(layer_idx)] if self.thresholds else 1e-6
            sparsity_masks = (activations <= 0)

            # Naive sparsity computation
            self.mlp_sparsity[layer_idx].append(
                sparsity_masks.float().mean().item()
            )

            abs_activations = activations.abs().cpu()
            k = max(1, int(0.1 * abs_activations.shape[-1]))
            _, top_indices = torch.topk(abs_activations, k)

            for idx in top_indices:
                self.heavy_hitters[layer_idx][idx] += 1
            
        # Clear GPU tensors from capture to free memory
        self.model.activation_capture.clear_captures()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()

        self.num_seqs += batch_size